<a href="https://colab.research.google.com/github/iannickgagnon/notebooks/blob/main/design_pattern_decorator_strategy_visitor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
from __future__ import annotations

from random import randint, choice

from dataclasses import dataclass, asdict

from typing import Protocol, runtime_checkable

# Generic Knapsack object class

In [19]:
@dataclass
class KnapsackObject:
  """
  Generic Knapsack object class.

  Attributes:
    weight (int): Weight of the object.
    value (int): Value of the object.
  """
  weight: int
  value: int

  def __post__init__(self):
    """
    Validate attributes after initialization.
    """
    if self.weight <= 0:
      raise ValueError("Weight must be > 0")

    if self.value < 0:
      raise ValueError("Value cannot be negative")

  def accept(self, visitor: KnapsackObjectVisitor):
    """
    Visitor pattern method for double dispatching.
    """
    raise NotImplementedError("The accept method is only implemented in the concrete subclasses.")

# Concrete KnapsckObject classes

In [20]:
class Television(KnapsackObject):
  """
  Concrete KanpsackObject class that corresponds to a television.

  Attributes:
    name (str): Name of the object.
  """
  name:str = "Television"

  def __init__(self, weight: int, value: int):
    """"
    Calls the parent class constructor.
    """
    super().__init__(weight=weight, value=value)

  def accept(self, visitor: KnapsackObjectVisitor):
    """
    Double dispatching (i.e., depens on KnapsackObject and KnapsackObjectVisitor).
    """
    return visitor.visit_television(self)


class Vase(KnapsackObject):
  """
  Concrete KnapsackObject class that corresponds to a vase.
  """
  name:str  = "Vase"

  def __init__(self, weight: int, value: int):
    """
    Calls the parent class constructor.
    """
    super().__init__(weight=weight, value=value)

  def accept(self, visitor: KnapsackObjectVisitor):
    """
    Double dispatching (i.e., depens on KnapsackObject and KnapsackObjectVisitor).
    """
    return visitor.visit_vase(self)

# Concrete visitors

In [21]:
@runtime_checkable
class KnapsackObjectVisitor(Protocol):
  """
  Visitor interface for KnapsackObject elements.

  Protocol defines a structural interface, which amounts to duck typing
  with type checking.

  That means:
    - A class does NOT need to inherit from `KnapsackObjectVisitor` to be
      considered a valid visitor.
    - It only needs to provide the same methods with compatible signatures.

  Example:

    class MyVisitor:
      def visit_television(self, obj: Television) -> float: ...
      def visit_vase(self, obj: Vase) -> float: ...

    `MyVisitor` is accepted anywhere a `KnapsackObjectVisitor` is expected,
    because it has the right method names/signatures.

  This is useful for the Visitor pattern because you can add new visitors
  (new operations) without modifying existing object classes, and without
  forcing visitors to inherit from a common base class.

  Normally, `Protocol` is meant for static type checkers (mypy/pyright).

  At runtime, Python does not enforce protocol conformance automatically.

  Decorating the Protocol with @runtime_checkable enables runtime checks like:

      isinstance(visitor, KnapsackObjectVisitor)

  With the decorator:
    - `isinstance` will return True if the object has the required attributes
      (here: `visit_television` and `visit_vase`) with a compatible shape.

    - It’s still not perfect "full type checking" at runtime; it’s mainly an
      structural attribute-presence check, not a guarantee about argument
      and return types.

  Without the decorator:
    - isinstance(visitor, KnapsackObjectVisitor) raises a TypeError.

  The `...` is the literal Python `Ellipsis` object.

  In a Protocol, these method bodies are stubs:
    - They say “this method must exist” and define its signature.
    - They do not provide an implementation.

  The type checker reads these stubs as abstract requirements.

  With this Protocol, the `accept()` methods can be typed as:
      def accept(self, visitor: KnapsackObjectVisitor) -> float:
          ...

  Then any new visitor class (InverseWeightVisitor, ValueDensityVisitor, etc.)
  will type-check as long as it defines the required methods with no inheritance
  required.
  """
  def visit_television(self, obj: Television) -> float: ...
  def visit_vase(self, obj: Vase) -> float: ...

class InverseWeightVisitor:
  """
  Returns the inverse of the object's weight.
  """
  def visit_television(self, obj: Television) -> float:
    return 1.0 / obj.weight

  def visit_vase(self, obj: Vase) -> float:
    return 1.0 / obj.weight

class ValueDensityVisitor:
  """
  Returns the value density (value/weight) of the object.
  """
  def visit_television(self, obj: Television) -> float:
    return obj.value / obj.weight

  def visit_vase(self, obj: Vase) -> float:
    return obj.value / obj.weight

# Data generator

In [24]:
@dataclass
class DataGenerator:
  """
  Generates a list of random objects for the Knapsack Problem.

  Attributes:
    weight_min_television_lb (int): Minimum weight of a television in pounds (lb).
    weight_max_television_lb (int): Maximum weight of a television in pounds (lb).
    value_min_television (int): Minimum value of a television in dollars.
    value_max_television (int): Maximum value of a television in dollars.
    weight_min_vase_lb (int): Minimum weight of a vase in pounds (lb).
    weight_max_vase_lb (int): Maximum weight of a vase in pounds (lb).
    value_min_vase (int): Minimum value of a vase in dollars.
    velue_max_vase (int): Maximum value of a vase in dollars.
  """
  weight_min_television_lb: int = 50
  weight_max_television_lb: int = 300
  value_min_television: int = 100
  value_max_television: int = 1500
  weight_min_vase_lb: int = 1
  weight_max_vase_lb: int = 10
  value_min_vase: int = 5
  value_max_vase: int = 3000

  def __post__init__(self):
    """
    Ensures that attributes (i.e. weights and values) are greater than zero.
    """
    accumulator: list[str] = []
    for attribute, value in asdict(self).items():
      if value <= 0:
        accumulator.append(attribute)
    if accumulator:
      raise ValueError(f"The following attributes must be greater than zero : {accumulator}")


  def generate(self, nb_objects: int = 10, verbose: bool = True) -> list[KnapsackObject]:
    """
    Generates a list of nb_objects random objects.

    Args:
      nb_objects (int, optional): The number of objects to generate. Defaults to 10.
      verbose (bool, optional): Whether to print progress or not. Defaults to True.

    Returns:
      (tuple[KnapsackObject]): A list of random KnapsackObject instances.

    Raises:
      ValueError: When the supplied number of objects is less than zero.
    """

    # Validate number of objects
    if nb_objects < 0:
      raise ValueError("The number of objects to generate must be >= 0")

    # Initialize return container
    objects: list[KnapsackObject] = []

    # Generate random objects
    for i in range(nb_objects):

      obj_factory: tuple = choice([(Television,
                            self.weight_min_television_lb,
                            self.weight_max_television_lb,
                            self.value_min_television,
                            self.value_max_television),
                            (Vase,
                            self.weight_min_vase_lb,
                            self.weight_max_vase_lb,
                            self.value_min_vase,
                            self.value_max_vase)])

      weight_min: int = obj_factory[1]
      weight_max: int = obj_factory[2]
      value_min: int = obj_factory[3]
      value_max: int = obj_factory[4]

      random_weight: int = randint(weight_min, weight_max)
      random_value: int = randint(value_min, value_max)

      random_object: KnapsackObject = obj_factory[0](weight=random_weight,
                                                     value=random_value)

      objects.append(random_object)

      # Show progress
      if verbose:
        print(f"Added object no.{i + 1}: {random_object}")

    return objects

# Run generator
objects = DataGenerator().generate()

Added object no.1: Vase(weight=6, value=677)
Added object no.2: Television(weight=112, value=1005)
Added object no.3: Vase(weight=6, value=146)
Added object no.4: Vase(weight=7, value=2071)
Added object no.5: Television(weight=239, value=1365)
Added object no.6: Vase(weight=9, value=2318)
Added object no.7: Vase(weight=9, value=2723)
Added object no.8: Vase(weight=3, value=2883)
Added object no.9: Vase(weight=5, value=255)
Added object no.10: Vase(weight=1, value=2252)


# Do something with the visitors
Calculate the inverse weights of the objects in the list.

In [28]:
inverse_weight_visitor = InverseWeightVisitor()
value_density_visitor = ValueDensityVisitor()

inverse_weights: list[float] = []
value_densities: list[float] = []
for knapsack_obj in objects:
  inverse_weights.append(knapsack_obj.accept(inverse_weight_visitor))
  value_densities.append(knapsack_obj.accept(value_density_visitor))

for weight, density in zip(inverse_weights, value_densities):
  print(f"weight (lb)          = {weight}\n"
        f"value density ($/lb) = {density}\n")

weight (lb)          = 0.16666666666666666
value density ($/lb) = 112.83333333333333

weight (lb)          = 0.008928571428571428
value density ($/lb) = 8.973214285714286

weight (lb)          = 0.16666666666666666
value density ($/lb) = 24.333333333333332

weight (lb)          = 0.14285714285714285
value density ($/lb) = 295.85714285714283

weight (lb)          = 0.0041841004184100415
value density ($/lb) = 5.7112970711297075

weight (lb)          = 0.1111111111111111
value density ($/lb) = 257.55555555555554

weight (lb)          = 0.1111111111111111
value density ($/lb) = 302.55555555555554

weight (lb)          = 0.3333333333333333
value density ($/lb) = 961.0

weight (lb)          = 0.2
value density ($/lb) = 51.0

weight (lb)          = 1.0
value density ($/lb) = 2252.0

